# AI Frontier · Day 2 · Hands-on Python

**Mr. K. Akshay Kumar** · Python & AI App Dev at Kovan Labs

---

**How this works:** I'll unlock one snippet at a time on the session website. When a snippet unlocks, copy it into your own Colab notebook (or run it here in this master copy), hit Shift+Enter, and watch.

By 12:30 PM you'll have written code that runs on your phone.


## Snippet 0 · Warm up the AI engines

_Setup_

Run this the SECOND you open the notebook. It downloads the AI models we'll use later (~800MB) while we do the intro. By the time we need them, they're ready.


In [ ]:
# Install compatible versions (mediapipe locked to 0.10.21 for Python 3.12)
!pip install -q --upgrade "numpy<2.0" "scipy<1.13" rembg deepface mediapipe==0.10.21 gradio qrcode onnxruntime tf-keras
import os

print("⏳ Pre-warming AI models in the background...")

# ---- Background removal model ----
from rembg import remove
from PIL import Image
_ = remove(Image.new('RGB', (10, 10), 'white'))
print("✅ Background-removal model ready")

# ---- Sentiment model ----
from transformers import pipeline
_clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
_ = _clf('warmup')
print("✅ Sentiment model ready")

# ---- DeepFace models ----
from deepface import DeepFace
os.makedirs('/tmp', exist_ok=True)
Image.new('RGB', (224, 224), color='white').save('/tmp/dummy.jpg')
try:
    _ = DeepFace.analyze('/tmp/dummy.jpg', actions=['age', 'gender', 'emotion'], enforce_detection=False)
except:
    pass
print("✅ Face-analysis models ready")

print("\n🚀 Warm-up complete!")


---
### Warning: Restart required before continuing

After the install cell above finishes, go to **Runtime - Restart session** in the top menu, then click **Run All** (Ctrl+F9).

**Why?** Google Colab boots with NumPy 2.x already loaded in memory. This session pins numpy<2.0 so MediaPipe 0.10.21 works correctly. The downgrade lands on disk but the running Python process still holds the old version in RAM. Any import that touches NumPy C extensions crashes with: ValueError: numpy.dtype size changed, may indicate binary incompatibility.

A runtime restart flushes memory and loads the pinned version fresh. The second run of the install cell is fast - packages are already cached.

---

## Snippet 1 · Python is just a really clean way to handle stuff

_Block 3 — Solve real problems_

Variables, lists, loops — but using filenames you'll actually recognize from your gallery.


In [ ]:
# These are typical phone photo filenames
photos = [
    "IMG_20240315_143201.jpg",
    "IMG_20240315_143215.jpg",
    "IMG_20240315_184422.jpg",
    "IMG_20240316_090101.jpg",
    "PXL_20240316_120304567.jpg",
    "Screenshot_2024-03-16-10-12-44.png",
    "IMG-20240316-WA0007.jpg",  # WhatsApp photo
    "IMG-20240316-WA0011.jpg",
]

# How many?
print(f"You have {len(photos)} photos\n")

# Loop and number them
for i, photo in enumerate(photos, 1):
    print(f"{i}. {photo}")

# Filter — just the WhatsApp ones (one line!)
whatsapp = [p for p in photos if "WA" in p]
print(f"\nWhatsApp photos only ({len(whatsapp)}): {whatsapp}")

# 🔥 Bonus: count screenshots vs camera shots
screenshots = [p for p in photos if p.startswith("Screenshot")]
camera = [p for p in photos if p.startswith("IMG_") or p.startswith("PXL_")]
print(f"\n📸 {len(camera)} camera shots | 📱 {len(screenshots)} screenshots")


## Snippet 2 · Rename your phone photos by date taken

_Block 3 — Solve real problems_

The script you'll use for the rest of your life. Reads EXIF metadata, renames every file to YYYY-MM-DD_HH-MM-SS.jpg.


**Before running:** In the left sidebar of Colab, click the 📁 folder icon, then create a folder called `photos`. Drag 5–10 photos from your phone/laptop into it.

In [ ]:
import os
from PIL import Image
from PIL.ExifTags import TAGS

FOLDER = '/content/photos'

def get_date_taken(path):
    # Read DateTimeOriginal from a photo's EXIF metadata
    try:
        exif = Image.open(path)._getexif()
        if not exif:
            return None
        for tag_id, value in exif.items():
            if TAGS.get(tag_id) == 'DateTimeOriginal':
                return value  # e.g., '2024:03:15 14:32:01'
    except Exception:
        return None

renamed = 0
skipped = 0
for fname in sorted(os.listdir(FOLDER)):
    old_path = os.path.join(FOLDER, fname)
    if not os.path.isfile(old_path):
        continue

    date = get_date_taken(old_path)
    if not date:
        print(f"⏭️  {fname}  (no EXIF date — probably a screenshot)")
        skipped += 1
        continue

    ext = os.path.splitext(fname)[1].lower()
    base = date.replace(':', '-').replace(' ', '_')
    new_name = f"{base}{ext}"
    new_path = os.path.join(FOLDER, new_name)

    # Collision-safe: if two photos taken same second, append a counter
    counter = 1
    while os.path.exists(new_path) and new_path != old_path:
        new_name = f"{base}_{counter}{ext}"
        new_path = os.path.join(FOLDER, new_name)
        counter += 1

    os.rename(old_path, new_path)
    print(f"✅ {fname}  →  {new_name}")
    renamed += 1

print(f"\n🎉 Renamed {renamed} photos, skipped {skipped}.")
print("👀 Check the photos/ folder in the sidebar — names have changed.")


## Snippet 3 · Find duplicate photos hiding in your folder

_Block 3 — Solve real problems_

How many copies of the same meme are in your WhatsApp Images right now? Let's find out.


In [ ]:
import os
import hashlib
from collections import defaultdict

FOLDER = '/content/photos'

def file_hash(path, chunk_size=8192):
    # MD5 fingerprint of a file's contents
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

# Bucket files by their content hash
buckets = defaultdict(list)
for fname in os.listdir(FOLDER):
    path = os.path.join(FOLDER, fname)
    if os.path.isfile(path):
        buckets[file_hash(path)].append(fname)

# Anything with more than one file in the same bucket = duplicate
dupes_found = False
for h, files in buckets.items():
    if len(files) > 1:
        dupes_found = True
        print(f"🔁 Duplicates: {files}")

if not dupes_found:
    print("✨ Clean folder — no duplicates!")


## Snippet 4 · Remove the background from a selfie

_Block 3 — Solve real problems_

Three lines of Python. A neural network does the work. You just used AI without knowing it.


**Before running:** Upload one selfie to Colab (drag it into the `/content` folder in the sidebar) and rename it `selfie.jpg`.

In [ ]:
from rembg import remove
from PIL import Image
from IPython.display import display

INPUT = '/content/photos/selfie.jpg'
OUTPUT = '/content/photos/selfie_nobg.png'

with Image.open(INPUT) as img:
    output = remove(img)        # ← the magic happens here
    output.save(OUTPUT)

print("📥 Before:")
display(Image.open(INPUT).resize((300, 300)))
print("\n📤 After (transparent background):")
display(output.resize((300, 300)))
print(f"\n✅ Saved to {OUTPUT}")


## Snippet 5 · Scrape the top headlines on Hacker News

_Block 4 — Reach the internet_

Your data isn't on your laptop — it's on the web. Python can grab any of it.


In [ ]:
import requests
from bs4 import BeautifulSoup

# Fetch the page (pretending to be a normal browser)
response = requests.get(
    'https://news.ycombinator.com/',
    headers={'User-Agent': 'Mozilla/5.0'}
)

# Parse the HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Grab every story title link
titles = soup.select('.titleline > a')

print("📰 Hacker News — right now:\n")
for i, t in enumerate(titles[:10], 1):
    print(f"{i:>2}. {t.text}")


## Snippet 6 · Highest-grossing Tamil films — straight into a DataFrame

_Block 4 — Reach the internet_

Remember Pandas from Day 1? Watch ONE LINE pull every table off a Wikipedia page.


In [ ]:
import pandas as pd

URL = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_Tamil_films'

# A User-Agent header pretends we are a browser — avoids Wikipedia's 403 Forbidden
storage_options = {'User-Agent': 'Mozilla/5.0'}

# pandas + BeautifulSoup under the hood — every <table> becomes a DataFrame
tables = pd.read_html(URL, storage_options=storage_options)
print(f"📊 Found {len(tables)} tables on the page.\n")

# Peek at the first column of each so we can pick the right one
for i, t in enumerate(tables[:6]):
    cols = list(t.columns)[:4]
    print(f"  Table {i}: {cols}")


In [ ]:
# Pick the table that has the actual gross list
# (if 0 isn't right, try 1, 2, ... based on the output above)
df = tables[0]
df.head(15)


## Snippet 7 · Browser automation (the Tatkal-booking superpower)

_Block 4 — Reach the internet_

DON'T run this in Colab — it needs a real screen. This is what runs on your laptop while you sleep.


This is for your **laptop**, not Colab. Skim it — we'll watch a recording instead. The same pattern automates IRCTC, college portals, Instagram, anything with a browser.

In [ ]:
# Run this on YOUR LAPTOP, not in Colab.
# Install once:  pip install playwright && playwright install chromium

from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)   # headless=False = you watch it
    page = browser.new_page()

    page.goto('https://www.irctc.co.in')
    page.fill('input[name="userId"]', 'your_username')
    page.fill('input[name="password"]', 'your_password')
    page.click('button:has-text("SIGN IN")')

    # ... fill journey form, click search, click book, fill passenger details...
    # Tatkal opens at 10:00 AM — bot beats every human in the queue.

    browser.close()


## Snippet 8 · Text sentiment — is this tweet happy or angry?

_Block 5 — Python IS AI_

Three lines of code, one of the most-used AI models on Earth. Day 3 (ML) and Day 6 (NLP) will show you how it works inside.


In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis',
                      model='distilbert-base-uncased-finetuned-sst-2-english')

texts = [
    "I'm having the best day of my life!",
    "This pizza is the worst thing I've ever eaten.",
    "The movie was okay, nothing special.",
    "Tomorrow's exam — I'm so ready to fail.",
]

for text in texts:
    result = classifier(text)[0]
    emoji = '😊' if result['label'] == 'POSITIVE' else '😞'
    print(f"{emoji}  {result['label']:<8} {result['score']:.1%}   \"{text}\"")


## Snippet 9 · Map your face — 468 AI landmarks live

_Block 5 — Python IS AI_

Take a webcam photo right inside Colab. MediaPipe finds your face and pins 468 points to it. This is what Snapchat and Instagram filters use.


**Step 1:** Run the webcam helper cell below (just once — it's a 30-line script you don't need to read).

In [ ]:
# WEBCAM HELPER — just run this cell once, then forget it exists
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = '📸 Capture';
            capture.style.cssText = 'padding:12px 24px;font-size:16px;background:#0a7;color:#fff;border:0;border-radius:6px;cursor:pointer;margin:8px;';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.maxWidth = '480px';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print("📷 Webcam helper loaded. Now run the next cell to take a photo.")


**Step 2:** Run the cell below. Your webcam opens — click 📸 Capture, smile, and watch.

In [ ]:
import mediapipe as mp
import cv2
from google.colab.patches import cv2_imshow

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

try:
    # Snap a selfie
    filename = take_photo('selfie.jpg')
    img = cv2.imread(filename)

    # Let MediaPipe find every contour of your face
    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1) as fm:
        results = fm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if results.multi_face_landmarks:
            for landmarks in results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    img, landmarks, mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing.DrawingSpec(
                        color=(0, 255, 0), thickness=1, circle_radius=1)
                )

    print("🧠 Your face — as seen by AI (468 landmarks):")
    cv2_imshow(img)

except NameError:
    print("❌ The webcam helper (take_photo) isn't loaded. Run the cell above first!")
except Exception as e:
    print(f"❌ An error occurred: {e}")


## Snippet 10 · What does AI think about your face? (age, gender, emotion)

_Block 5 — Python IS AI_

DeepFace runs three deep-learning models on your selfie and tells you your age, gender, and what you're feeling. Brace for laughter.


In [ ]:
from deepface import DeepFace
from PIL import Image
from IPython.display import display

# Analyze the same selfie from Snippet 9
result = DeepFace.analyze(
    'selfie.jpg',
    actions=['age', 'gender', 'emotion'],
    enforce_detection=False
)

r = result[0]

print(f"🎂 Estimated age: ~{r['age']}")
print(f"👤 Gender:        {r['dominant_gender']}")
print(f"😊 Emotion:       {r['dominant_emotion']}")
print(f"\nFull emotion breakdown:")
for emo, score in r['emotion'].items():
    bar = '█' * int(score / 2)
    print(f"   {emo:<10} {score:>5.1f}%  {bar}")

print()
display(Image.open('selfie.jpg').resize((300, 300)))


## Snippet 11 · Turn your code into a website your friend can hit from their phone

_Block 6 — Talk to the world_

Four lines of Gradio = a real FastAPI backend + auto-generated frontend + a public URL. Your phone scans the QR. You just shipped an app.


In [ ]:
import gradio as gr
from deepface import DeepFace

def analyze_face(image_path):
    if image_path is None:
        return "Please upload a photo!"
    try:
        result = DeepFace.analyze(
            image_path,
            actions=['age', 'gender', 'emotion'],
            enforce_detection=False
        )
        r = result[0]
        return (
            f"🎂 Age: ~{r['age']}\n"
            f"👤 Gender: {r['dominant_gender']}\n"
            f"😊 Emotion: {r['dominant_emotion']}"
        )
    except Exception as e:
        return f"Couldn't analyze that photo: {e}"

demo = gr.Interface(
    fn=analyze_face,
    inputs=gr.Image(type='filepath', label='📸 Snap or upload a selfie'),
    outputs=gr.Textbox(label='🤖 AI says...'),
    title='Face Analyzer — built in 10 minutes',
    description='Built by a 2nd-year student on Day 2 of the AI Frontier internship.'
)

# share=True opens a public URL via Gradio's tunnel (FastAPI under the hood)
demo.launch(share=True)


**After launching:** the cell prints a `*.gradio.live` URL. Copy it, paste it into the cell below to get a QR code you can put on a projector.

In [ ]:
import qrcode
from PIL import Image
from IPython.display import display

# Paste the gradio.live URL printed above
SHARE_URL = 'https://PASTE-YOUR-URL-HERE.gradio.live'

qr = qrcode.make(SHARE_URL)
qr.save('/content/qr.png')

print(f"🌐 Live URL: {SHARE_URL}")
print("📱 Scan with your phone:")
display(Image.open('/content/qr.png').resize((400, 400)))
